In [ ]:
df['date'] = pd.to_datetime(df['Month'], format= "%b-%y")

In [1]:
pip install graphviz

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install mlxtend

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 13.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   --------- ------------------------------ 1.8/8.1 MB 8.6 MB/s eta 0:00:01
   ----------------------- ---------------- 4.7/8.1 MB 11.0 MB/s eta 0:00:01
   --------------------------------- ------ 6.8/8.1 MB 11.3 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 10.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   -------- ------------------------------- 2.6/12.3 MB 13.3 MB/s eta 0:00:01
   --------------------- ------------------ 6.6/12.3 MB 15.8 MB/s eta 0:00:01
   ---------------------------- ----------- 8.9/12.3 MB 14.8 MB/s eta 0:00:01
   --------------------------------- ------ 10.2/12.3 MB 12.2 MB/s eta 0:00:01
   --------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.0 requires numpy<2.2,>=1.24, but you have numpy 2.4.2 which is incompatible.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.8.0 which is incompatible.
streamlit 1.45.1 requires pandas<3,>=1.4.0, but you have pandas 3.0.0 which is incompatible.


In [15]:
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth, association_rules
from graphviz import Digraph

class FPTreeNode:
    def __init__(self, name, count, parent):
        self.name = name
        self.count = count
        self.parent = parent
        self.children = {}
        self.link = None


class FPTree:
    def __init__(self):
        self.root = FPTreeNode("Root", 0, None)
        self.headers = {}

    def add_transaction(self, transaction):
        current_node = self.root
        for item in transaction:
            if item in current_node.children:
                current_node.children[item].count += 1
            else:
                new_node = FPTreeNode(item, 1, current_node)
                current_node.children[item] = new_node
                # update headers
                if item not in self.headers:
                    self.headers[item] = new_node
                else:
                    #Maintain linked list of same items
                    last_node = self.headers[item]
                    while last_node.link is not None:
                        last_node = last_node.link
                    last_node.link = new_node
                current_node = current_node.children[item]

    def visualize(self):
        dot = Digraph()
        self._add_nodes(dot, self.root)
        return dot

    def _add_nodes(self, dot, node):
        for child_name, child_node in node.children.items():
            dot.node(str(child_node), f"{child_node.count} ({child_node.count})")
            dot.edge(str(node), str(child_node))
            self._add_nodes(dot, child_node)

data = {
    'Transaction_ID': [1,2,3,4,5],
    'Milk': [1,1,0,1,0],
    'Bread': [1,0,1,1,1],
    'Butter': [1,1,0,1,0],
    'Eggs': [0,1,1,0,1],
    'Cheese': [0,1,1,1,0]
}

df = pd.DataFrame(data).set_index('Transaction_ID')

df_bool = df.astype(bool)

frequent_itemsets = fpgrowth(df_bool, min_support=0.4, use_colnames=True)

rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.6)

item_support = df_bool.sum().sort_values(ascending=False)
sorted_items = item_support.index.tolist()

sorted_transactions = []

for index, row in df_bool.iterrows():
    sorted_row = [item for item in sorted_items if row[item]]
    sorted_transactions.append(sorted_row)

#constyruct FP tree
fp_tree = FPTree()
for transaction in sorted_transactions:
    fp_tree.add_transaction(transaction)

# fp_tree_graph = fp_tree.visualize()
# fp_tree_graph.render("FP_Tree", format="png", cleanup=True)

print("Frequent itemsets:")
print(frequent_itemsets)

print('\nAssociation Rules:')
print(rules)

from IPython.display import Image
Image(filename='FP_Tree.png')

Frequent itemsets:
    support                           itemsets
0       0.8                 frozenset({Bread})
1       0.6                frozenset({Butter})
2       0.6                  frozenset({Milk})
3       0.6                frozenset({Cheese})
4       0.6                  frozenset({Eggs})
5       0.4         frozenset({Butter, Bread})
6       0.4        frozenset({Butter, Cheese})
7       0.6          frozenset({Butter, Milk})
8       0.4           frozenset({Milk, Bread})
9       0.4          frozenset({Milk, Cheese})
10      0.4   frozenset({Butter, Milk, Bread})
11      0.4  frozenset({Butter, Milk, Cheese})
12      0.4         frozenset({Cheese, Bread})
13      0.4          frozenset({Eggs, Cheese})
14      0.4           frozenset({Eggs, Bread})

Association Rules:
                    antecedents                  consequents  \
0           frozenset({Butter})           frozenset({Bread})   
1           frozenset({Butter})          frozenset({Cheese})   
2           froze

C:\ProgramData\anaconda3\Lib\site-packages\executing\executing.py:713: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
  right=ast.Str(s=sentinel),
C:\ProgramData\anaconda3\Lib\ast.py:602: DeprecationWarning: Constant.__init__ got an unexpected keyword argument 's'. Support for arbitrary keyword arguments is deprecated and will be removed in Python 3.15.
  return Constant(*args, **kwargs)
C:\ProgramData\anaconda3\Lib\ast.py:602: DeprecationWarning: Attribute s is deprecated and will be removed in Python 3.14; use value instead
  return Constant(*args, **kwargs)
C:\ProgramData\anaconda3\Lib\ast.py:602: DeprecationWarning: Constant.__init__ missing 1 required positional argument: 'value'. This will become an error in Python 3.15.
  return Constant(*args, **kwargs)
C:\ProgramData\anaconda3\Lib\site-packages\executing\executing.py:713: DeprecationWarning: ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instea

FileNotFoundError: [Errno 2] No such file or directory: 'FP_Tree.png'

In [21]:
years = range(1995,1998)

date  = pd.to_datetime(years, format='%Y')

In [22]:
print(date)

DatetimeIndex(['1995-01-01', '1996-01-01', '1997-01-01'], dtype='datetime64[us]', freq=None)


In [ ]:
23/Jan/2025

pd.to_datetime(x, format='%d/%b/%Y')